# 06.6 — Confidence routing

A good decoder doesn't just predict — it knows *when it's likely wrong*. The Optimal model has two **confidence heads** that output a self-assessed certainty: **Trial confidence** (am I sure about this trial?) and **Task confidence** (am I sure about this task/dimension?). They're combined into a `TotalConfidence` that modulates the classification loss — so the model is penalized less for errors it flagged as uncertain, and more for confident mistakes. This notebook is about the two heads and how they route.

This notebook covers:

* Why a model should estimate its own confidence.
* Trial vs Task confidence — what each head sees and outputs.
* `TotalConfidence = Task × Trial` — the conjunction.
* The `ConfidenceHistory` state and `apply_confidence_routing`.

**Prerequisites:** [04.6 multi-head classifier](../04_architecture/04.6_multi_head_classifier.ipynb), [06.4 EMA priors](06.4_the_ema_prior_normalization_deep_dive.ipynb).

## Section 1 — What MATLAB does

`cgg_lossConfidence` and `cgg_addTaskConfidenceToClassifier` add confidence heads whose outputs gate the loss:

```matlab
TrialConfidence = trialConfidenceHead(latent);       % per-trial certainty
TaskConfidence  = taskConfidenceHead(classifier);    % per-task certainty
TotalConfidence = TaskConfidence .* TrialConfidence;  % conjunction

% The confidence MODULATES the classification loss (interpolated CE, 06.11)
% and has its OWN loss driving it toward calibration (06.7 controller).
```

`ConfidenceType = ["Trial", "Task"]` in the Optimal config activates both. `WeightConfidence` scales the confidence loss's contribution. The port implements the full kernel — five subtleties in Critical Note #29.

## Section 2 — The Python concepts you need

### 2.1 — Why self-estimated confidence

Neural decoding is noisy — some trials are genuinely ambiguous. A model that outputs a *calibrated* confidence lets downstream analysis weight or filter predictions ("use only high-confidence trials"). And during training, confidence-modulated loss means the model isn't forced to be certain on genuinely-hard trials — it can say "unsure" and be penalized less, focusing its capacity on the learnable cases. Confidence is both an output (useful downstream) and a training signal (a better loss).

### 2.2 — Trial vs Task confidence

Two heads, two granularities:

| Head | Reads | Estimates | Analogy |
|---|---|---|---|
| **Trial** | the latent representation | is THIS trial's signal clear? | "the recording looks clean" |
| **Task** | the classifier's penultimate features | is THIS dimension decodable? | "this feature is generally easy" |

Trial confidence varies trial-to-trial (a noisy trial → low); Task confidence is more about the dimension's inherent difficulty. Both output a value in (0, 1) — a probability-like certainty.

### 2.3 — The conjunction: `TotalConfidence = Task × Trial`

In [ ]:
import torch

# Confidence values in (0, 1) — high = certain, low = unsure.
trial_conf = torch.tensor([0.9, 0.5, 0.2, 0.8])    # per-trial certainty
task_conf  = torch.tensor([0.7, 0.7, 0.7, 0.3])    # per-task certainty

total_conf = task_conf * trial_conf                 # conjunction (AND-like)
print("trial:", trial_conf.tolist())
print("task: ", task_conf.tolist())
print("total:", [round(v, 3) for v in total_conf.tolist()])
print("→ TotalConfidence is high only when BOTH streams are confident (a logical AND).")

The multiplicative conjunction means the model is "totally confident" only when *both* the trial looks clean AND the task is decodable — either doubt drags the total down. This is stricter (and more honest) than either head alone: a clean trial of an undecodable dimension still gets low total confidence. The Optimal config uses both heads; using only one (`ConfidenceType=["Trial"]`) skips the conjunction.

### 2.4 — The confidence history state

In [ ]:
from neural_data_decoding.training.losses.confidence import ConfidenceHistory

# Like LossPriors (06.4), confidence keeps running EMA state threaded through training:
hist = ConfidenceHistory.initial()
print("initial history — total:", hist.total.item(), "trial:", hist.trial.item(),
      "task:", hist.task.item(), "beta:", hist.beta.item())
print("all start at 1.0 (MATLAB's empty-history initialization).")

`ConfidenceHistory` carries four running scalars: EMA of each confidence stream (total/trial/task) plus the controller's `beta` ([06.7](06.7_the_confidence_pd_controller.ipynb)). Like the loss priors ([06.4 §2.4](06.4_the_ema_prior_normalization_deep_dive.ipynb)) and the checkpoint bookkeeping ([05.5](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb)), it's *detached* state threaded through the epoch — updated each batch, carried to the next, never backpropped through.

### 2.5 — Routing it all together

`apply_confidence_routing` is the full kernel: it takes the classifier output, the targets, and both confidence streams, and returns the confidence loss plus the updated history. It also drives the interpolated cross-entropy ([06.11](06.11_single_total_loss_three_subnetworks.ipynb)) that lets confidence modulate the classification loss. The five subtleties (Critical Note #29) — confidence dropout, batch correction, dataset-vs-batch confidence, the interpolation, and the P-controller — are what make it intricate; [06.7](06.7_the_confidence_pd_controller.ipynb) and [06.9](06.9_per_batch_prior_correction.ipynb) unpack the trickiest.

## Section 3 — The `neural_data_decoding` implementation

`ConfidenceHistory` — the detached EMA + controller state:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/confidence.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.strip() == "total: torch.Tensor")
for k in range(i, min(i + 5, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* Four scalar-tensor fields: `total`, `trial`, `task`, `beta` — the running confidence EMAs plus the controller state ([06.7](06.7_the_confidence_pd_controller.ipynb)).
* `initial()` seeds everything to 1.0 — MATLAB's empty-history behavior (`cgg_lossConfidence.m` lines 98-109), so the first batch isn't dividing by uninitialized state.
* The module docstring enumerates the five Critical Note #29 subtleties; the `apply_confidence_routing` function comments map each block to its subtlety — a heavily-annotated high-risk port.
* All state is detached ([02.5 §2.4](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)) — the confidence *value* carries gradient (the head must learn), but the running *history* is a constant.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the AND behavior

Show that `TotalConfidence` is low whenever EITHER stream is low, using a small grid of trial/task values.

In [ ]:
# Reveal:
for t in (0.2, 0.9):
    for k in (0.2, 0.9):
        total = t * k
        verdict = "confident" if total > 0.5 else "unsure"
        print(f"  trial={t}, task={k} → total={total:.2f}  ({verdict})")
print("→ only trial=0.9 AND task=0.9 clears 0.5 — either doubt suppresses the total.")

### Exercise 4.2 — history is detached

Confirm the confidence history tensors don't require grad (they're running state, not trainable).

In [ ]:
# Reveal:
hist = ConfidenceHistory.initial()
for name in ("total", "trial", "task", "beta"):
    t = getattr(hist, name)
    print(f"  {name}: requires_grad = {t.requires_grad}")
print("→ all False: the history is a running constant, updated per batch, never backpropped.")

## Section 5 — Common errors

### Confidence collapses to 0 or 1 (no calibration)

Without the P-controller ([06.7](06.7_the_confidence_pd_controller.ipynb)) driving it toward a target, confidence can degenerate — always-certain (ignore the signal) or always-unsure (game the modulated loss). The controller is what keeps it calibrated.

### Using one head but expecting the conjunction

`TotalConfidence = Task × Trial` needs both heads. `ConfidenceType=["Trial"]` gives only trial confidence — no conjunction. Check the config lists both.

### History not threaded through epochs

Like the loss priors ([06.4 §5](06.4_the_ema_prior_normalization_deep_dive.ipynb)), the confidence history must be carried batch-to-batch. A fresh `ConfidenceHistory.initial()` each batch resets the EMA and the controller — the calibration never accumulates.

### Confidence loss backprops into the history

The history is detached state (§3). Only the confidence *head output* carries gradient. Backpropping the history would corrupt the running estimate.

### Task confidence head errors — classifier has no penultimate features

Task confidence reads the classifier's penultimate layer, which `MultiHeadClassifier` (Logistic) doesn't expose but `DeepLSTMClassifier` does ([04.6 §2.4](../04_architecture/04.6_multi_head_classifier.ipynb)). Task confidence needs the deeper classifier.

## Section 6 — Further reading

- [What uncertainties do we need in deep learning? (Kendall & Gal)](https://arxiv.org/abs/1703.04977) — the aleatoric/epistemic framing behind confidence.
- [`src/neural_data_decoding/training/losses/confidence.py`](../../src/neural_data_decoding/training/losses/confidence.py) — the full routing kernel.
- [06.7 the confidence PD-controller](06.7_the_confidence_pd_controller.ipynb) — the mechanism that calibrates it.

Next up: **[06.7 the confidence PD-controller](06.7_the_confidence_pd_controller.ipynb)** — the highest-risk port in the whole codebase.